# Recipe 06 — Context Window
> Work directly with the `ContextWindow` for fine-grained token management.

| | |
|---|---|
| **Module** | `anchor.models` |
| **Key classes** | `ContextWindow`, `ContextItem`, `SourceType` |
| **Difficulty** | Intermediate |

In [ ]:
from anchor.models import ContextWindow, ContextItem, SourceType, QueryBundle

## 1 — Create a window
`ContextWindow` tracks a hard token ceiling. Items are added explicitly.

In [ ]:
window = ContextWindow(max_tokens=4096)
print(f"Capacity       : {window.max_tokens} tokens")
print(f"Used           : {window.used_tokens}")
print(f"Remaining      : {window.remaining_tokens}")

## 2 — Add a single item
Each `ContextItem` carries its own `token_count`.

In [ ]:
item = ContextItem(
    id="doc-1",
    content="Context engineering is the discipline of curating the right "
            "information for an LLM's context window.",
    source=SourceType.RETRIEVAL,
    score=0.95,
    priority=5,
    token_count=20,
)
window.add_item(item)

print(f"Items          : {len(window.items)}")
print(f"Used tokens    : {window.used_tokens}")
print(f"Utilization    : {window.utilization:.1%}")

## 3 — Add multiple items
Add a batch of items at once.

In [ ]:
batch = [
    ContextItem(id="doc-2", content="RAG combines retrieval with generation.",
               source=SourceType.RETRIEVAL, score=0.85, priority=3, token_count=12),
    ContextItem(id="doc-3", content="Token budgets prevent context overflow.",
               source=SourceType.RETRIEVAL, score=0.70, priority=1, token_count=10),
    ContextItem(id="mem-1", content="User asked about context engineering.",
               source=SourceType.MEMORY, score=None, priority=8, token_count=10),
]
for b in batch:
    window.add_item(b)

print(f"Total items    : {len(window.items)}")
print(f"Used tokens    : {window.used_tokens}")

## 4 — Priority-based insertion
`add_items_by_priority` sorts items by priority and fills until the budget
is exhausted — higher-priority items go first.

In [ ]:
window2 = ContextWindow(max_tokens=50)  # tight budget

candidates = [
    ContextItem(id="lo", content="Low priority filler text",
               source=SourceType.RETRIEVAL, score=0.5, priority=1, token_count=20),
    ContextItem(id="hi", content="Critical info",
               source=SourceType.RETRIEVAL, score=0.9, priority=10, token_count=20),
    ContextItem(id="mid", content="Moderately useful",
               source=SourceType.RETRIEVAL, score=0.7, priority=5, token_count=20),
]

window2.add_items_by_priority(candidates)

print(f"Included {len(window2.items)} of {len(candidates)} candidates")
for item in window2.items:
    print(f"  {item.id:5s}  priority={item.priority}  tokens={item.token_count}")

## 5 — Remaining capacity check
Always check `remaining_tokens` before manual insertion to avoid overflow.

In [ ]:
print(f"Window 1 remaining : {window.remaining_tokens}")
print(f"Window 2 remaining : {window2.remaining_tokens}")

# Safe insertion pattern
new_item = ContextItem(
    id="safe", content="Extra info",
    source=SourceType.RETRIEVAL, score=0.6, priority=2, token_count=15,
)
if new_item.token_count <= window.remaining_tokens:
    window.add_item(new_item)
    print(f"Added '{new_item.id}' — now {window.used_tokens} tokens used")
else:
    print(f"Not enough room for '{new_item.id}'")

## Key Takeaways
- `ContextWindow(max_tokens=N)` enforces a hard token ceiling.
- `add_items_by_priority()` auto-fills highest-priority items first.
- Always check `remaining_tokens` before manual adds.

**Next:** [Enrichers](07_enrichers.ipynb)